In [1]:
# ============================================================
# CELL 0 – LOCAL ENVIRONMENT SETUP
# ============================================================
import os, sys

# Cấu trúc Local-first: lấy thư mục gốc của project
# Vì notebook nằm trong thư mục notebooks/, nên PROJECT_ROOT là thư mục cha.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
REPO_PATH = PROJECT_ROOT

# Add src/ to Python path
if os.path.join(PROJECT_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

print('✅ Local Environment ready. PROJECT_ROOT:', PROJECT_ROOT)


✅ Local Environment ready. PROJECT_ROOT: d:\000MINHTHONG\Junior - Semester II\ANLPB\FinalProject\hate-speech-detection


In [2]:
# ============================================================
# CELL 1 – PREPROCESSING LOGIC
# ============================================================
import re
import os
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

def clean_text(text):
    if not isinstance(text, str): return ""
    # 1. Remove URLs and HTML entities
    text = re.sub(r'http\S+|www\S+|&[a-z]+;', '', text)
    # 2. Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # 3. Normalize whitespace
    # (CRITICAL: No lowercase, No accent removal as per pipeline rules)
    text = ' '.join(text.split())
    return text

def process_data(file_name):
    # Construct paths using DRIVE_ROOT and REPO_PATH setup from Cell 0
    input_path = os.path.join(PROJECT_ROOT, f"data/raw/vihsd/{file_name}.csv")
    output_path = os.path.join(PROJECT_ROOT, f"data/processed/{file_name}.parquet")

    if not os.path.exists(input_path):
        print(f"⚠️ Warning: File not found {input_path}")
        return None

    df = pd.read_csv(input_path)
    # Use 'free_text' column from the raw CSV
    df['text'] = df['free_text'].apply(clean_text)

    # Filter very short samples (at least 2 characters)
    df = df[df['text'].str.len() >= 2]

    # Save as parquet (Enterprise requirement for efficiency)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df[['text', 'label_id']].to_parquet(output_path, index=False)
    print(f"✅ Processed and saved: {output_path}")
    return df

# Execute processing for all splits
train_clean = process_data('train')
dev_clean = process_data('dev')
test_clean = process_data('test')

# Calculate Class Weights to handle data imbalance in later phases
if train_clean is not None:
    labels = train_clean['label_id'].unique()
    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.sort(labels),
        y=train_clean['label_id']
    )
    class_weights_dict = dict(zip(np.sort(labels), weights))
    print(f"\n📊 Calculated Class Weights: {class_weights_dict}")

    # Decision: Save these weights to a config or result for Phase 5
    import json
    with open(os.path.join(PROJECT_ROOT, "results/class_weights.json"), 'w') as f:
        json.dump({int(k): float(v) for k, v in class_weights_dict.items()}, f)
    print("✅ Class weights saved to results/class_weights.json")

✅ Processed and saved: d:\000MINHTHONG\Junior - Semester II\ANLPB\FinalProject\hate-speech-detection\data/processed/train.parquet
✅ Processed and saved: d:\000MINHTHONG\Junior - Semester II\ANLPB\FinalProject\hate-speech-detection\data/processed/dev.parquet
✅ Processed and saved: d:\000MINHTHONG\Junior - Semester II\ANLPB\FinalProject\hate-speech-detection\data/processed/test.parquet

📊 Calculated Class Weights: {0: 0.4033414092469211, 1: 4.978816199376947, 2: 3.1263693270735526}
✅ Class weights saved to results/class_weights.json
